# Infinity — independent TPU experiment (v5e-1)

**Open this notebook separately. Do not change the runtime type in your running GPU notebook.** Choose **Runtime → Change runtime type → TPU v5e-1**, then run the cells in order. Colab may limit simultaneous runtimes or free TPU availability; it could disconnect either session.

This is an experimental TPU adaptation, not yet tested on a real TPU. It starts from the same preserved public checkpoint as the GPU notebook (step 536,897; 68,722,816 sampled positions), uses the same fixed dataset and original training cycle, and saves a **separate** copy of the model, AdamW optimizer and random state in `MyDrive/Infini-Colab-Parallel-TPU-books100m-43698262111`. It does not overwrite the GPU copy or change the GitHub run. The three copies diverge; none are automatically merged or deployed.

The initial TPU/XLA compilation and downloads may take several minutes. The default training cell runs four 15-minute segments automatically, saving after each. Time limits are approximate and compilation, validation and Drive copying can extend them. Free runtimes can terminate; at most the current unsaved segment is ordinarily lost, but a failed or delayed Drive backup can lose more. Only run one TPU session per backup folder. Confirm backups appear in Drive. Each uses about 0.9 GB; keep older backups until you know the newest is good, then remove old ones manually if needed. TPU and GPU arithmetic differ, so results are not bitwise identical or necessarily better.

Only load checkpoints you trust: these are pickle files. Allow roughly 10 GB or more local disk space for dependencies and checkpoint files, plus Drive storage. Installation may prompt you to restart the TPU runtime; if so, restart and rerun all cells.


In [ ]:
import os, sys, subprocess, importlib.metadata
if 'torch' in sys.modules or 'torch_xla' in sys.modules:
    raise RuntimeError('PyTorch was already imported. Restart the runtime, then run all cells from the top.')
os.environ['PJRT_DEVICE'] = 'TPU'
# Official matching PyTorch/XLA stable versions. The TPU extra includes libtpu.
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'torch==2.8.0', 'torch_xla[tpu]==2.8.0'], check=True)
# Probe in a separate process, so the notebook kernel does not retain the TPU.
probe = "import torch, torch_xla, torch_xla.runtime as r; assert r.device_type() == 'TPU', 'Select TPU v5e-1 in Runtime settings'; print('TPU ready:', torch_xla.device(), 'PyTorch', torch.__version__)"
subprocess.run([sys.executable, '-c', probe], check=True, env=os.environ.copy())
import torch, numpy
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')
BASE = Path('/content/infini-colab-tpu')
REPO = BASE / 'code'
DATA = BASE / 'datasets' / 'books100m'
RUN = BASE / 'run'
BACKUP = Path('/content/drive/MyDrive/Infini-Colab-Parallel-TPU-books100m-43698262111')
for directory in (BASE, RUN, BACKUP): directory.mkdir(parents=True, exist_ok=True)
print('TPU backups:', BACKUP)


## Download and verify the fixed training sources and data

The preserved checkpoint download is about 900 MB. The model and dataset code are pinned to the commit used by the GitHub training; the separate TPU adapter is checked against its embedded SHA-256. No GitHub credentials are needed and nothing is written to GitHub.


In [ ]:
import hashlib, urllib.request
TRAIN_COMMIT = '1c33e90d654ad48b0c96102474943dbd1af1f0ae'
CKPT_SHA = 'f29f0de64a6ab1b20f260f05e8d9c141e1ae1c9786b789568ac13ce70e3a3426'
DATA_SHA = 'bc73cd70757479c21da2bbbcfcd0d36b3c374ed126a35668cb0c295c274d30e3'
RELEASE = 'https://github.com/Yuvii123-bot/InFiNi/releases/download/'

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        while True:
            chunk = f.read(8 * 1024 * 1024)
            if not chunk: break
            h.update(chunk)
    return h.hexdigest()

def download_checked(url, destination, expected):
    if destination.exists() and sha256_file(destination) == expected:
        print('Verified cached', destination.name); return
    temporary = destination.with_name(destination.name + '.download')
    try:
        request = urllib.request.Request(url, headers={'User-Agent': 'infini-colab'})
        h = hashlib.sha256()
        with urllib.request.urlopen(request, timeout=120) as src, open(temporary, 'wb') as dst:
            while True:
                chunk = src.read(8 * 1024 * 1024)
                if not chunk: break
                dst.write(chunk); h.update(chunk)
        if h.hexdigest() != expected: raise ValueError('Checksum mismatch: ' + destination.name)
        os.replace(temporary, destination)
    finally:
        if temporary.exists(): temporary.unlink()
    print('Downloaded and verified', destination.name)

if not REPO.exists():
    subprocess.run(['git','clone','--no-checkout','https://github.com/Yuvii123-bot/InFiNi.git',str(REPO)], check=True)
    subprocess.run(['git','-C',str(REPO),'checkout','--detach',TRAIN_COMMIT], check=True)
head = subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'], text=True).strip()
if head != TRAIN_COMMIT: raise RuntimeError('Unexpected training code commit: ' + head)
# Do not run modified training code under the original commit identity.
if subprocess.check_output(['git','-C',str(REPO),'status','--porcelain','--','llm'], text=True).strip():
    raise RuntimeError('Local training code was modified; use a clean runtime.')
checkpoint = BASE / 'checkpoint-16-1.pt'
archive = BASE / 'dataset-books100m.tar.gz'
download_checked(RELEASE + 'infinity-qa75m-44205300285/checkpoint-16-1.pt', checkpoint, CKPT_SHA)
download_checked(RELEASE + 'infinity-books100m-data-43697791940/dataset-books100m.tar.gz', archive, DATA_SHA)
# Overwrite the local working dataset with the verified fixed archive on each setup.
(BASE / 'datasets').mkdir(parents=True, exist_ok=True)
subprocess.run(['tar', '-xzf', str(archive), '-C', str(BASE / 'datasets')], check=True)
if not (DATA / 'dataset.json').is_file(): raise RuntimeError('Dataset archive did not contain the expected manifest')
ADAPTER = BASE / 'train_colab_tpu.py'
download_checked('https://github.com/Yuvii123-bot/InFiNi/raw/refs/heads/arena/034eb1ca-infini/scripts/train_colab_tpu.py', ADAPTER,
                 '10a9af7066eefe92bb48be18ce149bde4cc224603fecf0602635508db2103c83')
print('Training code:', head, '| Fixed dataset:', DATA, '| TPU adapter SHA verified')


## Run the TPU segments

The latest verified **TPU** backup in the TPU folder is resumed; otherwise the preserved initial checkpoint is used. Progress messages show sampled token positions per second including compilation, which is not a controlled GPU benchmark. The first segment may be slow while XLA compiles. Each segment must finish training, pass finite validation, and complete Drive backup before the next starts. If anything fails, the previous verified Drive backup remains untouched.

After four segments, rerun this cell to continue. After a disconnect, select a TPU again and run all cells. Ask for help if installation, TPU allocation, compilation or saving fails; do not switch the GPU notebook's runtime.


In [ ]:
import json, re, shutil, time, math
# Only one TPU session may use this backup folder at a time.
SEGMENTS = 4
SECONDS_PER_SEGMENT = 900
assert 1 <= SEGMENTS <= 24 and 60 <= SECONDS_PER_SEGMENT <= 3600
pattern = re.compile(r'^checkpoint-step-(\d+)-([0-9a-f]{64})\.pt$')

def checkpoint_info(path):
    # Checkpoint loading is intentionally restricted to the pinned GitHub file or your trusted own backups.
    state = torch.load(path, map_location='cpu', weights_only=False, mmap=True)
    cycle = state.get('cycle') or {}
    if cycle.get('id') != 'books100m-v1' or cycle.get('start_step') != 500 or cycle.get('steps') != 781250:
        raise RuntimeError('Checkpoint belongs to a different training cycle')
    if not state.get('optimizer') or not state.get('np_rng') or state.get('torch_rng') is None:
        raise RuntimeError('Checkpoint is missing the optimizer or CPU random state')
    step = int(state['step'])
    if not 536897 <= step <= 781750: raise RuntimeError('Unexpected checkpoint step: ' + str(step))
    return step, int(state['tokens_seen'])

def backup_checkpoint(source):
    step, tokens = checkpoint_info(source)
    digest = sha256_file(source)
    destination = BACKUP / f'checkpoint-step-{step:09d}-{digest}.pt'
    if destination.exists():
        if sha256_file(destination) != digest: raise RuntimeError('Existing backup has a checksum mismatch')
    else:
        temporary = BACKUP / (destination.name + '.partial')
        try:
            shutil.copyfile(source, temporary)
            if sha256_file(temporary) != digest: raise RuntimeError('Drive backup verification failed')
            os.replace(temporary, destination)
        finally:
            if temporary.exists(): temporary.unlink()
    if sha256_file(destination) != digest: raise RuntimeError('Published Drive backup verification failed; previous backups are unchanged')
    print(f'Verified Drive backup: step {step:,}, total sampled positions {tokens:,}\n{destination}', flush=True)
    return step

local_checkpoint = RUN / 'ckpt.pt'
pending = RUN / 'segment-in-progress'
if pending.exists():
    # The last segment or Drive copy did not finish. Never silently promote its local checkpoint.
    if local_checkpoint.exists():
        quarantine = RUN / f'ckpt-unverified-{time.time_ns()}.pt'
        os.replace(local_checkpoint, quarantine)
        print('Unverified interrupted segment retained locally:', quarantine)
    pending.unlink()
    print('Resuming from the most recent verified Drive backup (or the original snapshot).')
backups = sorted(((int(m.group(1)), p, m.group(2)) for p in BACKUP.iterdir() if (m := pattern.fullmatch(p.name))), reverse=True)
# On rerun, choose the newest verified local or Drive state, never silently replace newer progress.
local_step = checkpoint_info(local_checkpoint)[0] if local_checkpoint.exists() else -1
if len(backups) > 1 and backups[0][0] == backups[1][0] and backups[0][2] != backups[1][2]:
    raise RuntimeError('Multiple different backups have the same newest step; choose one manually before continuing.')
if backups and backups[0][0] == local_step:
    if sha256_file(backups[0][1]) != backups[0][2]: raise RuntimeError('Newest Drive backup is corrupted; inspect it before continuing.')
    if sha256_file(local_checkpoint) != backups[0][2]:
        raise RuntimeError('Local and Drive checkpoints differ at the same step; choose one manually before continuing.')
if backups and backups[0][0] > local_step:
    step, restore, expected = backups[0]
    if sha256_file(restore) != expected: raise RuntimeError('Newest Drive backup is corrupted; inspect it before continuing.')
    if checkpoint_info(restore)[0] != step: raise RuntimeError('Backup filename and checkpoint disagree.')
    temporary = RUN / 'ckpt.restore'
    shutil.copyfile(restore, temporary)
    if sha256_file(temporary) != expected: raise RuntimeError('Restored copy checksum mismatch')
    os.replace(temporary, local_checkpoint)
elif local_step < 0:
    if sha256_file(checkpoint) != CKPT_SHA or checkpoint_info(checkpoint)[0] != 536897: raise RuntimeError('Initial checkpoint verification failed')
    shutil.copyfile(checkpoint, local_checkpoint)
current_step = checkpoint_info(local_checkpoint)[0]
if current_step > 536897 and (not backups or current_step > backups[0][0]):
    print('Saving locally recovered progress to Drive before continuing.')
    backup_checkpoint(local_checkpoint)
print('Resuming separate TPU copy at step', current_step)

for segment in range(SEGMENTS):
    before, _ = checkpoint_info(local_checkpoint)
    if before >= 781750:
        print('The original 100M-position cycle is complete for this Colab copy.'); break
    cmd = [sys.executable, str(ADAPTER), '--data', str(DATA), '--run', str(RUN), '--max-seconds', str(SECONDS_PER_SEGMENT)]
    env = os.environ.copy()
    env['PJRT_DEVICE'] = 'TPU'
    env['PYTHONPATH'] = str(REPO) + os.pathsep + env.get('PYTHONPATH', '')
    print(f'Segment {segment+1}/{SEGMENTS}: training on TPU from step {before:,}...', flush=True)
    pending.write_text('A segment or its Drive backup is not yet verified.\n')
    result = subprocess.run(cmd, cwd=REPO, env=env)
    if result.returncode != 0: raise RuntimeError('Training failed; prior Drive backups are unchanged. Exit ' + str(result.returncode))
    status = json.loads((RUN / 'status.json').read_text())
    if status.get('device') != 'xla' or status.get('runtime') != 'TPU' or not status.get('optimizer_state_restored') or not str(status.get('source', '')).startswith('resumed '):
        raise RuntimeError('Trainer did not confirm TPU and optimizer restore; refusing to save as a successful continuation.')
    if status.get('val_loss') is None or not math.isfinite(status['val_loss']):
        raise RuntimeError('Validation loss is not finite; prior Drive backups are unchanged.')
    after = checkpoint_info(local_checkpoint)[0]
    if int(status.get('step', -1)) != after: raise RuntimeError('Status and checkpoint steps disagree.')
    if after <= before: raise RuntimeError('No training progress; prior Drive backups are unchanged.')
    backup_checkpoint(local_checkpoint)
    pending.unlink()
    if after >= 781750:
        print('The original 100M-position cycle is complete for this Colab copy.'); break
print('Done. Run this cell again to continue. The GPU notebook and GitHub job were not touched or replaced.')
